# Import Libraries

In [1]:
!pip install -q sentence-transformers qdrant-client tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 9.2 MB/s eta 0:00:00


In [19]:
import json
import pickle
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from google.colab import userdata
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from google import genai

# Upload File

In [4]:
with open("/content/ar-tafsir-qurtubi.json", "r", encoding="utf-8") as f:
    tafsir_data = json.load(f)

print(len(tafsir_data))

114


# Chunk Text

In [5]:
def chunk_text(text, max_words=200, overlap=30):
    words = text.split()
    if len(words) <= max_words:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
    return chunks

In [6]:
records = []

for surah_idx, surah_ayat in enumerate(tafsir_data, start=1):
    for ayah_idx, ayah_text in enumerate(surah_ayat, start=1):
        if not ayah_text or not ayah_text.strip():
            continue
        chunks = chunk_text(ayah_text)
        for c_idx, chunk in enumerate(chunks):
            records.append({
                "surah": surah_idx,
                "ayah": ayah_idx,
                "chunk_index": c_idx,
                "text": chunk,
            })

print("chunks:", len(records))

chunks: 17335


# Embedding

In [7]:
model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
texts_to_embed = ["passage: " + r["text"] for r in records]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [8]:
batch_size = 64
all_embeddings = []

for i in tqdm(range(0, len(texts_to_embed), batch_size)):
    batch = texts_to_embed[i:i + batch_size]
    emb = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embeddings.extend(emb)

100%|██████████| 271/271 [17:13<00:00,  3.81s/it]


# Save Res

In [9]:
for r, emb in zip(records, all_embeddings):
    r["embedding"] = emb.tolist()

with open("/content/tafsir_embeddings.pkl", "wb") as f:
    pickle.dump(records, f)

print("total embeddings:", len(records))
print("each embedding:", len(records[0]["embedding"]))

total embeddings: 17335
each embedding: 1024


# Read Secrets

In [11]:
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

# Upload Embedding

In [12]:
with open("/content/tafsir_embeddings.pkl", "rb") as f:
    records = pickle.load(f)

print(len(records))

17335


# Connect to Qdrant

In [13]:
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
COLLECTION_NAME = "tafsir_qurtubi"

# Create collection

In [14]:
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

/tmp/ipykernel_1192/1407422365.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

# Upload Data

In [15]:
points = []
for i, r in enumerate(records):
    points.append(
        PointStruct(
            id=i,
            vector=r["embedding"],
            payload={
                "surah": r["surah"],
                "ayah": r["ayah"],
                "chunk_index": r["chunk_index"],
                "text": r["text"],
            },
        )
    )

batch_size = 256
for i in range(0, len(points), batch_size):
    batch = points[i:i + batch_size]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"اترفع {min(i + batch_size, len(points))} / {len(points)}")

print(len(points))


اترفع 256 / 17335
اترفع 512 / 17335
اترفع 768 / 17335
اترفع 1024 / 17335
اترفع 1280 / 17335
اترفع 1536 / 17335
اترفع 1792 / 17335
اترفع 2048 / 17335
اترفع 2304 / 17335
اترفع 2560 / 17335
اترفع 2816 / 17335
اترفع 3072 / 17335
اترفع 3328 / 17335
اترفع 3584 / 17335
اترفع 3840 / 17335
اترفع 4096 / 17335
اترفع 4352 / 17335
اترفع 4608 / 17335
اترفع 4864 / 17335
اترفع 5120 / 17335
اترفع 5376 / 17335
اترفع 5632 / 17335
اترفع 5888 / 17335
اترفع 6144 / 17335
اترفع 6400 / 17335
اترفع 6656 / 17335
اترفع 6912 / 17335
اترفع 7168 / 17335
اترفع 7424 / 17335
اترفع 7680 / 17335
اترفع 7936 / 17335
اترفع 8192 / 17335
اترفع 8448 / 17335
اترفع 8704 / 17335
اترفع 8960 / 17335
اترفع 9216 / 17335
اترفع 9472 / 17335
اترفع 9728 / 17335
اترفع 9984 / 17335
اترفع 10240 / 17335
اترفع 10496 / 17335
اترفع 10752 / 17335
اترفع 11008 / 17335
اترفع 11264 / 17335
اترفع 11520 / 17335
اترفع 11776 / 17335
اترفع 12032 / 17335
اترفع 12288 / 17335
اترفع 12544 / 17335
اترفع 12800 / 17335
اترفع 13056 / 17335
اترفع 13312 / 17335
ات

# Try

In [17]:
model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [18]:
def search_tafsir(question, top_k=3):
    query_vector = model.encode("query: " + question, normalize_embeddings=True)
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    ).points
    for hit in results:
        p = hit.payload
        print(f"\n--- سورة {p['surah']} آية {p['ayah']} (score: {hit.score:.3f}) ---")
        print(p["text"][:400])
search_tafsir("ما معنى الرفث في الحج؟")



--- سورة 2 آية 197 (score: 0.868) ---
، فكأنه قال : رجل فرض . وقال : فيهن ولم يقل فيها ، فقال قوم : هما سواء في الاستعمال . وقال المازني أبو عثمان : الجمع الكثير لما لا يعقل يأتي كالواحدة المؤنثة ، والقليل ليس كذلك ، تقول : الأجذاع انكسرن ، والجذوع انكسرت ، ويؤيد ذلك قول الله تعالى : إن عدة الشهور ثم قال : منها .السادسة : قوله تعالى : فلا رفث قال ابن عباس وابن جبير والسدي وقتادة والحسن وعكرمة والزهري ومجاهد ومالك : الرفث الجماع ، أي ف

--- سورة 22 آية 29 (score: 0.866) ---
قوله تعالى : ثم ليقضوا تفثهم أي ثم ليقضوا بعد نحر الضحايا والهدايا ما بقي عليهم من أمر الحج ؛ كالحلق ، ورمي الجمار ، وإزالة شعث ، ونحوه . قال ابن عرفة : أي ليزيلوا عنهم أدرانهم . وقال الأزهري : التفث الأخذ من الشارب ، وقص الأظفار ، ونتف الإبط ، وحلق العانة ؛ وهذا عند الخروج من الإحرام . وقال النضر بن شميل : التفث في كلام العرب إذهاب الشعث وسمعت الأزهري يقول : التفث في كلام العرب لا يعرف إلا من قول

--- سورة 22 آية 29 (score: 0.860) ---
ما أتفثك ؛ أي ما أوسخك وأقذرك . قال أمية بن أبي الصلت :ساخين آباطهم لم يقذفوا تفث

# Read Gemini Secret

In [20]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# Connect

In [21]:
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

COLLECTION_NAME = "tafsir_qurtubi"

In [22]:
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Search for Qdrant

In [23]:
def retrieve_tafsir(question, top_k=5):
    query_vector = embed_model.encode("query: " + question, normalize_embeddings=True)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    ).points
    return results

# Prompt

In [28]:
def ask_tafsir_bot(question, top_k=5):
    hits = retrieve_tafsir(question, top_k=top_k)

    if not hits:
        return "معلش، مالقتش تفسير مرتبط بسؤالك."

    context_parts = []
    for hit in hits:
        p = hit.payload
        context_parts.append(
            f"[سورة رقم {p['surah']} - آية رقم {p['ayah']}]\n{p['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""أنت مساعد متخصص في تفسير القرآن الكريم، تعتمد فقط على تفسير القرطبي.

السياق التالي هو مقتطفات من تفسير القرطبي مرتبطة بسؤال المستخدم:

{context}

سؤال المستخدم: {question}

المطلوب منك:
- أجب على السؤال بالاعتماد فقط على النصوص أعلاه، ولا تخترع معلومات من عندك.
- اذكر رقم السورة والآية اللي استندت إليها في إجابتك.
- لو النصوص أعلاه ما فيهاش إجابة كافية للسؤال، قل ذلك بوضوح بدل التخمين.
- اكتب الإجابة بلغة عربية فصيحة وواضحة ومباشرة.
"""
    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )
    return response.text

# Try

In [29]:
answer = ask_tafsir_bot("ما معنى الرفث في الحج؟")
print(answer)

بناءً على تفسير القرطبي الوارد في النص لـ **[سورة رقم 2 - آية رقم 197]** في قوله تعالى: ﴿فَلا رَفَثَ﴾، تنوعت عبارات المفسرين في معنى "الرفث" في الحج على عدة أقوال:

1. **الجماع:** وهو قول ابن عباس، وابن جبير، والسدي، وقادتة، والحسن، وعكرمة، والزهري، ومجاهد، ومالك؛ أي: فلا جماع لأنه يفسد الحج.
2. **الإفحاش للمرأة بالكلام:** وهو قول عبد الله بن عمر، وطاوس، وعطاء وغيرهم، ورواية عن ابن عباس؛ ومثّلوا له بأن يقول للمرأة صراحة من غير كناية: "إذا أحللنا فعلنا بك كذا".
3. **الإفحاش بذكر النساء:** وقيل هو التكلم بالفحش بذكر النساء، سواء كان ذلك بحضرتهن أم في غيبتهن.
4. **ما يريده الرجل من أهله:** وقيل إن الرفث كلمة جامعة لكل ما يريده الرجل من أهله.
5. **اللغوا من الكلام:** وهو قول أبي عبيدة.

*(المرجع: تفسير القرطبي - سورة رقم 2 - آية رقم 197)*
